# HulagwAI — model training (Google Colab)
**Before you run anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

Everything lives in the repo folder `ml-service/` (Python files in `training/`). This notebook only calls them, so the repo stays the single source of truth.
Full instructions: `ml-service/training/README.md`.

## 1 · Mount Google Drive (so work survives a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hulagwai_work'   # all outputs go here

## 2 · Get the code
Either clone your repo, or upload `ml-service.zip` to Drive and unzip it.

In [ ]:
# Option A - clone (replace the URL; for a private repo use a token: https://<TOKEN>@github.com/<you>/<repo>.git)
# !git clone https://github.com/<you>/<repo>.git /content/repo
# %cd /content/repo/ml-service

# Option B - from Drive
# !unzip -q /content/drive/MyDrive/ml-service.zip -d /content/repo
# %cd /content/repo/ml-service

## 3 · Install training dependencies

In [ ]:
!pip install -q -r requirements-train.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 4 · Kaggle access (for CIFAKE / ArtiFact) and optional Hugging Face token
Kaggle ▸ *Settings ▸ API ▸ Create New Token* gives you a username and key. In Colab open the 🔑 **Secrets** panel, add `KAGGLE_USERNAME` and `KAGGLE_KEY`, and tick *Notebook access*.

Optional: huggingface.co ▸ *Settings ▸ Access Tokens ▸ Create new token* (type **Read**, free account is fine) and add it as secret `HF_TOKEN`. It gives the OpenFake download its own rate limit. Nothing is billed. If the token is rejected or its limit runs out, the download simply continues without it.

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Optional Hugging Face token (free account is enough): own rate limit for the OpenFake download instead of
# Colab's shared one. If it is missing, invalid or its limit is used up, the download continues without it.
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Hugging Face token found - OpenFake uses it (falls back to anonymous automatically)')
except Exception:
    os.environ.pop('HF_TOKEN', None)
    print('No HF_TOKEN secret - OpenFake downloads anonymously (works, just a lower shared rate limit)')

## 5 · (Optional) quick smoke test — 2-3 minutes
Proves the pipeline works before you spend hours. Uses tiny subsets and random weights, so the numbers mean nothing.

In [ ]:
# !python -m training.run_all --profile smoke --work /content/smoke --version 0.0.1

## 6 · Real training
`colab_fast` ≈ CIFAKE + ArtiFact + OpenFake (+ GenImage if you provide it). Allow roughly **2–4 hours** on a free T4 (estimate — depends on Colab load). Safe to re-run after a disconnect: finished stages are skipped, and the OpenFake / RAISE / Synthbuster downloads resume where they stopped. Stage 1 checks that every intended dataset actually made it into train/val/test (and the OOD set when requested). Only missing CIFAKE stops the run; any other gap (an optional dataset missing from val/test, too few families for attribution, a failed Synthbuster/RAISE download) is logged as a warning and training continues. Re-running retries the downloads.

**No GPU left?** Run the same cell on a CPU runtime anyway: it does only the stages that don't need a GPU (stage 1, all downloads, and stage 5, the frequency model), saves them to Drive and stops with a message. When your GPU quota is back (usually within a day), switch to *T4 GPU* and run the cell again; it continues with the GPU stages. (CIFAKE, ~100 MB, is re-downloaded automatically then because a new runtime starts with an empty local disk; ArtiFact and OpenFake images are kept on Drive and are not downloaded again.)

**Free Colab (~2.5 h GPU limit)?** Keep `PROFILE = 'colab_mini'` below (small per-dataset sizes, no total image cap, a 15-min OpenFake download filled evenly across generator families, 2 CNN epochs). Still too slow? Lower `SUBSET` (e.g. `0.5` = half of that data) or add `--cnn-epochs 1`. Each profile/subset gets its own Drive folder, so runs never mix. Tip: run the cell on a CPU runtime first so the downloads don't use GPU time.

Optional extras:
* `--genimage-root /content/drive/MyDrive/genimage` — a GenImage subset you downloaded (see `docs/DATASETS.md`)
* `--with-ood` — downloads **Synthbuster** (generators never seen in training) for the honest final test
* `--raise-csv /content/drive/MyDrive/RAISE_2k.csv` — RAISE is a CSV of download links (not a folder): 300 of its TIFFs (`--raise-n`) are downloaded as the real camera photos for that test
* `--ood-real-dir <folder>` — or a folder of your own real camera photos
* `--no-openfake` — skip the OpenFake download

In [ ]:
PROFILE = 'colab_mini'   # 'colab_mini' fits free Colab (~2.5 h); 'colab_fast' / 'colab_full' need more GPU time
SUBSET = 1.0             # fraction of the profile's data to train on, e.g. 0.5 = half (less data = faster, weaker)
WORK = f'/content/drive/MyDrive/hulagwai_work_{PROFILE}_{SUBSET}'   # one folder per data size (outputs are not mixed)
RAISE_CSV = '/content/drive/MyDrive/RAISE_2k.csv'   # upload the RAISE CSV here (or remove --raise-csv below)
!python -m training.run_all --profile $PROFILE --subset $SUBSET --work "$WORK" --version 1.0.0 --with-ood --raise-csv $RAISE_CSV

## 7 · Look at the result

In [ ]:
import json
m = json.load(open(f'{WORK}/reports/metrics.json'))
t = m['test']
print({k: round(t[k], 4) for k in ('accuracy','precision','recall','specificity','f1','roc_auc','mae','mse','brier','log_loss','ece')})
print('3-way verdict:', m['selective'])
print('robustness:', m['robustness'])
print('unseen generators:', m.get('ood'))

## 8 · Download the bundle
The zip is also in your Drive folder. Unzip it into **`ml-service/models/current/`** on your laptop.

In [ ]:
from google.colab import files
import glob
files.download(sorted(glob.glob(f'{WORK}/hulagwai-ensemble-*.zip'))[-1])